# Translation Pipeline — `additional_kanker_nl`

**Source**: `additional_cancer_information.csv`  
**Output**: `additional_cancer_information_translated.csv`  
**Trustworthiness** (from Makhotso's matrix): `4.9`  
**Volume**: 148 rows · ~686k Dutch chars · ~30 min

### How to run

1. Upload the source CSV to the same folder as this notebook (or change the path below)
2. Run all cells top to bottom
3. The output CSV will appear next to the source, with `_en` columns added

Each translated column is added as a sibling next to the Dutch original. Empty cells, URLs, hashes and dates are skipped automatically.

## 1. Install dependencies

In [ ]:
pip install --quiet deep-translator pandas

zsh:1: command not found: pip


## 2. Shared translation library

*(Inlined so the notebook is self-contained — same code as `translate_lib.py`.)*

In [2]:
"""
Shared translation library for CSN Team 23 — Dutch → English.

Design choices (from facilitator session + earlier IKNL pipeline):
- deep-translator (Google backend), free tier
- Sentence-aware chunking at 4500 chars (under Google's 5000-char cap)
- 0.5s polite delay between API calls
- Skip cells that are empty, URLs, hashes, dates, or pure numbers
- Cache translations on content_hash to avoid re-translating identical Dutch
- Add a `_en` column next to each translated Dutch column, preserving source structure
"""
import re
import time
import hashlib
from typing import List, Optional
import pandas as pd
from deep_translator import GoogleTranslator

MAX_CHUNK = 4500          # under Google's 5000-char limit
DELAY_SEC = 0.5           # polite delay
RETRY = 3                 # retries per chunk on transient failure

# In-memory cache: maps SHA-1 of Dutch text → English translation
_cache: dict = {}


def _is_translatable(text) -> bool:
    """Return True if the cell holds Dutch text worth translating."""
    if pd.isna(text):
        return False
    s = str(text).strip()
    if not s:
        return False
    # Skip URLs, hashes, ISO dates, pure numbers
    if s.startswith(("http://", "https://", "www.")):
        return False
    if re.fullmatch(r"[0-9a-f]{32,128}", s):  # content hash
        return False
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}.*", s):  # ISO date
        return False
    if re.fullmatch(r"[\d\s.,+\-%]+", s):  # numbers
        return False
    if len(s) < 3:
        return False
    return True


def _split_into_sentences(text: str) -> List[str]:
    """Split on sentence boundaries while preserving punctuation."""
    # Split on . ! ? followed by whitespace
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p]


def _chunk_text(text: str, max_chars: int = MAX_CHUNK) -> List[str]:
    """Group sentences into chunks ≤ max_chars."""
    if len(text) <= max_chars:
        return [text]
    sentences = _split_into_sentences(text)
    chunks: List[str] = []
    buf = ""
    for s in sentences:
        if len(buf) + len(s) + 1 <= max_chars:
            buf = (buf + " " + s).strip()
        else:
            if buf:
                chunks.append(buf)
            # Hard-split sentences that exceed the cap on their own
            if len(s) > max_chars:
                for i in range(0, len(s), max_chars):
                    chunks.append(s[i:i + max_chars])
                buf = ""
            else:
                buf = s
    if buf:
        chunks.append(buf)
    return chunks


def _translate_chunk(chunk: str) -> str:
    """Translate a single chunk with retries."""
    last_err = None
    for attempt in range(RETRY):
        try:
            return GoogleTranslator(source="nl", target="en").translate(chunk)
        except Exception as e:
            last_err = e
            time.sleep(1.0 * (attempt + 1))
    # All retries failed — return original with marker so we know
    return f"[TRANSLATION_FAILED: {last_err}] {chunk}"


def translate_text(text) -> Optional[str]:
    """Public API: translate one cell (string or NaN) → English or None."""
    if not _is_translatable(text):
        return text  # pass through unchanged
    s = str(text).strip()
    # Cache lookup
    key = hashlib.sha1(s.encode("utf-8")).hexdigest()
    if key in _cache:
        return _cache[key]
    # Chunk → translate → reassemble
    chunks = _chunk_text(s)
    translated_chunks = []
    for c in chunks:
        translated_chunks.append(_translate_chunk(c))
        time.sleep(DELAY_SEC)
    out = " ".join(translated_chunks)
    _cache[key] = out
    return out


def translate_dataframe(df: pd.DataFrame,
                        columns_to_translate: List[str],
                        progress_every: int = 10) -> pd.DataFrame:
    """
    Translate specified columns in df, adding `<col>_en` siblings.
    Preserves original Dutch columns untouched.
    """
    out = df.copy()
    total_cells = sum(out[c].notna().sum() for c in columns_to_translate)
    done = 0
    t0 = time.time()
    for col in columns_to_translate:
        en_col = f"{col}_en"
        translations = []
        for idx, val in out[col].items():
            translations.append(translate_text(val))
            done += 1
            if done % progress_every == 0:
                elapsed = time.time() - t0
                rate = done / max(elapsed, 1e-6)
                eta = (total_cells - done) / max(rate, 1e-6)
                print(f"  [{done}/{total_cells}] cells | "
                      f"elapsed {elapsed:.0f}s | ETA {eta:.0f}s | "
                      f"cache_size={len(_cache)}")
        out[en_col] = translations
    print(f"  ✅ Done {total_cells} cells in {time.time() - t0:.0f}s "
          f"(cache hits saved time on repeats)")
    return out


def get_cache_stats() -> dict:
    return {"cached_strings": len(_cache)}


## 3. Configure paths and columns

Change `SRC` if your file lives elsewhere. `COLUMNS` lists the Dutch-text columns to translate — based on the schema-fit validation we did before starting.

In [17]:
SRC = '/Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /CSV Folder/additional_cancer_information.csv'
DST = '/Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /Translation /Translated_output/additional_cancer_information_translated.csv'
COLUMNS = ['Page Title', 'Article Content']
TRUSTWORTHINESS = 4.9   # from Makhotso's source-selection matrix


## 4. Load the source CSV

In [6]:
import pandas as pd
df = pd.read_csv(SRC)
print(f'Loaded {SRC}: {df.shape[0]} rows × {df.shape[1]} cols')
print('Columns:', list(df.columns))
df.head(2)


Loaded /Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /CSV Folder/additional_cancer_information.csv: 148 rows × 6 cols
Columns: ['Global Track', 'Page Title', 'Source URL', 'Scrape Date', 'Content Hash', 'Article Content']


,Global Track,Page Title,Source URL,Scrape Date,Content Hash,Article Content
0,Fatigue & Consequences,Tips bij vermoeidheid,https://www.kanker.nl/gevolgen-van-kanker/verm...,2026-05-31 14:58:01,04bc3dbdd9f18bf39eb8086ad1776c4a9023a61967e28d...,[Delen]\n\nLotgenoten weten als geen ander hoe...
1,Fatigue & Consequences,Wat te doen tegen vermoeidheid,https://www.kanker.nl/gevolgen-van-kanker/verm...,2026-05-31 14:58:02,520b539880ded5b4f7a5cb3c9caeb882dcc3ad8d1705d0...,[Delen]\n\nChronische vermoeidheid is erg verv...


## 5. Translate Dutch → English

The pipeline:
1. **Filter** — skip empties, URLs, hashes, dates
2. **Chunk** — split long text on sentence boundaries (≤ 4500 chars per chunk)
3. **Translate** — call Google Translate via deep-translator
4. **Cache** — identical strings translated only once
5. **Reassemble** — chunks joined back into the cell
6. **Side-by-side** — Dutch column preserved; English added as `<col>_en`

In [7]:
translated = translate_dataframe(df, COLUMNS, progress_every=25)
print()
print(f'Output shape: {translated.shape}')
print(f'New columns added: {[c for c in translated.columns if c.endswith("_en")]}')


  [25/296] cells | elapsed 28s | ETA 302s | cache_size=20
  [50/296] cells | elapsed 57s | ETA 278s | cache_size=42
  [75/296] cells | elapsed 91s | ETA 268s | cache_size=66
  [100/296] cells | elapsed 118s | ETA 230s | cache_size=86
  [125/296] cells | elapsed 151s | ETA 207s | cache_size=109
  [150/296] cells | elapsed 181s | ETA 176s | cache_size=129
  [175/296] cells | elapsed 240s | ETA 166s | cache_size=149
  [200/296] cells | elapsed 289s | ETA 139s | cache_size=171
  [225/296] cells | elapsed 363s | ETA 114s | cache_size=196
  [250/296] cells | elapsed 429s | ETA 79s | cache_size=216
  [275/296] cells | elapsed 504s | ETA 38s | cache_size=239
  ✅ Done 296 cells in 559s (cache hits saved time on repeats)

Output shape: (148, 8)
New columns added: ['Page Title_en', 'Article Content_en']


## 6. Spot-check translation quality

In [8]:
for col in COLUMNS:
    en_col = f'{col}_en'
    if en_col not in translated.columns:
        continue
    sample = translated[[col, en_col]].dropna().head(1)
    if sample.empty:
        continue
    print('=' * 60)
    print(f'COLUMN: {col}')
    print(f'NL: {str(sample[col].iloc[0])[:200]}')
    print(f'EN: {str(sample[en_col].iloc[0])[:200]}')
    print()


COLUMN: Page Title
NL: Tips bij vermoeidheid
EN: Tips for fatigue

COLUMN: Article Content
NL: [Delen]

Lotgenoten weten als geen ander hoe het is om steeds maar moe te zijn. Hieronder lees je over lotgenotencontact en vind je tips van lotgenoten.

[Ervaringen uitwisselen]

Soms kan het prettig
EN: [Share]

Fellow sufferers know better than anyone what it is like to be tired all the time. Below you can read about contact with fellow sufferers and find tips from fellow sufferers.

[Exchange exper



## 7. Add trustworthiness column and save

Hard-codes the trustworthiness score (since none of the source files contained it). Saves a UTF-8 CSV with all original columns + new `_en` columns.

In [18]:
translated['trustworthiness'] = TRUSTWORTHINESS
translated.to_csv(DST, index=False, encoding='utf-8')
print(f'💾 Saved {DST}')
print(f'   Rows: {translated.shape[0]}')
print(f'   Cols: {translated.shape[1]}')
print(f'   Cache hits during this run: {get_cache_stats()}')


💾 Saved /Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /Translation /Translated_output/additional_cancer_information_translated.csv
   Rows: 148
   Cols: 9
   Cache hits during this run: {'cached_strings': 255}


## ✅ Done

Hand the resulting CSV to Quadry for inclusion in the consolidated `csn_knowledge` table.  
The Dutch columns are preserved alongside the English ones so any reviewer can spot-check translations.